<a href="https://colab.research.google.com/github/Diro1981/nyx-powerful-calculator/blob/main/stable_diffusion_v1_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U diffusers

## Local Inference on GPU
Model page: https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
pip install -U diffusers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 75.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [71]:
import torch
from diffusers import AutoPipelineForImage2Image
from diffusers.utils import load_image, make_image_grid
import os

# 1. Load the pipeline for GPU (uses float32 by default)
print("Loading pipeline...")
pipeline = AutoPipelineForImage2Image.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    use_safetensors=True
)
print("Pipeline loaded.")

# Optional: Send pipeline explicitly to GPU
pipeline = pipeline.to("cuda")
print("Pipeline moved to CUDA.")

# 2. Load initial image
url = "https://www.shutterstock.com/image-photo/beautiful-young-bob-haircut-woman-600nw-2742226433.jpg"
print(f"Loading initial image from URL: {url}")
init_image = load_image(url)
print(f"Initial image loaded. Type: {type(init_image)}")

# 3. Generate image
prompt = "cinematic, detailed, 8k"
negative_prompt = "deformed, ugly, disfigured, bad anatomy, low quality, poor quality"

print(f"Starting image generation with prompt: '{prompt}', negative prompt: '{negative_prompt}', strength: 0.65, and num_inference_steps: 50")

try:
    generated_result = pipeline(
        prompt,
        image=init_image,
        strength=0.65, # Reduced strength to better preserve original features
        negative_prompt=negative_prompt,
        num_inference_steps=50 # Increased inference steps for better quality
    )
    print(f"Pipeline call completed. Generated result type: {type(generated_result)}")
    print(f"Generated result content (first 100 chars): {str(generated_result)[:100]}...")

    if generated_result and hasattr(generated_result, 'images') and generated_result.images:
        image = generated_result.images[0]
        output_filename = "output.png"

        print(f"Image generated successfully. Attempting to save to: {os.getcwd()}/{output_filename}")

        try:
            image.save(output_filename)
            if os.path.exists(output_filename):
                print(f"Successfully saved {output_filename}.")
            else:
                print(f"Warning: {output_filename} was not found on disk after saving attempt.")
        except Exception as e:
            print(f"Error saving image '{output_filename}': {e}")
            print("Please check if the image object is valid and if there's enough disk space.")
    else:
        print("Image generation returned no images or an invalid result. Output might be empty.")
        if generated_result and hasattr(generated_result, 'images'):
            print(f"generated_result.images was: {generated_result.images}")
        else:
            print("generated_result or generated_result.images attribute was missing.")

except Exception as e:
    print(f"Error during image generation pipeline execution: {e}")
    print("Please check your input parameters and model setup. This could be due to invalid input or GPU memory issues.")


Loading pipeline...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Pipeline loaded.
Pipeline moved to CUDA.
Loading initial image from URL: https://www.shutterstock.com/image-photo/beautiful-young-bob-haircut-woman-600nw-2742226433.jpg
Initial image loaded. Type: <class 'PIL.Image.Image'>
Starting image generation with prompt: 'cinematic, detailed, 8k', negative prompt: 'deformed, ugly, disfigured, bad anatomy, low quality, poor quality', strength: 0.65, and num_inference_steps: 50


  0%|          | 0/32 [00:00<?, ?it/s]

Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


Pipeline call completed. Generated result type: <class 'diffusers.pipelines.stable_diffusion.pipeline_output.StableDiffusionPipelineOutput'>
Generated result content (first 100 chars): StableDiffusionPipelineOutput(images=[<PIL.Image.Image image mode=RGB size=448x600 at 0x7BA096CEF7D0...
Image generated successfully. Attempting to save to: /content/output.png
Successfully saved output.png.


## Text-to-Image Generation

To generate images from just a text prompt, we need to load a different type of diffusion pipeline: `AutoPipelineForText2Image`. This pipeline doesn't require an initial image as input.

In [74]:
import torch
from diffusers import AutoPipelineForText2Image

# Clear any cached GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA memory cache cleared.")

# 1. Load the text-to-image pipeline with half-precision (float16) to save memory
print("Loading text-to-image pipeline with float16...")
text2image_pipeline = AutoPipelineForText2Image.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16, # Use float16 for reduced memory usage
    use_safetensors=True
)
print("Text-to-image pipeline loaded.")

# Send the pipeline to GPU if available
# Enable model CPU offload to save GPU memory
text2image_pipeline.enable_model_cpu_offload()
print("Text-to-image pipeline moved to CUDA and CPU offload enabled.")


CUDA memory cache cleared.
Loading text-to-image pipeline with float16...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Text-to-image pipeline loaded.
Text-to-image pipeline moved to CUDA and CPU offload enabled.


Now, let's generate an image using only a text prompt. We can reuse the `prompt` and `negative_prompt` from before, or create new ones.

In [95]:
# 2. Define your prompt and negative prompt for text-to-image
text_prompt = "funny cartoon man pose"
text_negative_prompt = "deformed, ugly, ugly face, extra hands, extra legs, extra fringers, ugly eyes and mouth, disfigured, bad anatomy, low quality, poor quality, blurry, worst quality, noise, out of focus"

# Define image dimensions (reduced for better upscaling memory management)
image_width = 1080
image_height = 512

print(f"Generating image from text prompt: '{text_prompt}' with dimensions {image_width}x{image_height}")

try:
    # Generate image without an initial image
    generated_text_image_result = text2image_pipeline(
        text_prompt,
        negative_prompt=text_negative_prompt,
        num_inference_steps=70, # Using 70 steps for better quality
        width=image_width,
        height=image_height
    )

    if generated_text_image_result and hasattr(generated_text_image_result, 'images') and generated_text_image_result.images:
        text_generated_image = generated_text_image_result.images[0]
        text_output_filename = "text_generated_output.png"

        print(f"Text-to-image generated successfully. Attempting to save to: {os.getcwd()}/{text_output_filename}")

        try:
            text_generated_image.save(text_output_filename)
            if os.path.exists(text_output_filename):
                print(f"Successfully saved {text_output_filename}.")
            else:
                print(f"Warning: {text_output_filename} was not found on disk after saving attempt.")
        except Exception as e:
            print(f"Error saving text-generated image '{text_output_filename}': {e}")
            print("Please check if the image object is valid and if there's enough disk space.")
    else:
        print("Text-to-image generation returned no images or an invalid result. Output might be empty.")

except Exception as e:
    print(f"Error during text-to-image generation pipeline execution: {e}")
    print("Please check your input parameters and model setup. This could be due to invalid input or GPU memory issues.")

Generating image from text prompt: 'funny cartoon man pose' with dimensions 1080x512


  0%|          | 0/70 [00:00<?, ?it/s]

Text-to-image generated successfully. Attempting to save to: /content/text_generated_output.png
Successfully saved text_generated_output.png.


Once the `text_generated_output.png` file is created, you can download it using the `google.colab.files.download` command, similar to before.

In [96]:
from google.colab import files

# Download the newly generated image
# This cell will only work if text_generated_output.png was successfully created.
files.download('text_generated_output.png')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Upscaling Generated Images

To achieve truly higher resolution and improved detail, especially towards '4K' quality, it's generally recommended to use a dedicated upscaling model. These models are specifically trained to take a lower-resolution image and enhance it, adding details and increasing its size without just stretching the pixels.

We will use `StableDiffusionUpscalePipeline` for this purpose. It takes a smaller image (like the one we just generated) and upscales it, in this case by a factor of 4.

In [47]:
import torch
from diffusers import StableDiffusionUpscalePipeline

# Clear any cached GPU memory before loading a new pipeline
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA memory cache cleared for upscaler.")

# Explicitly unload the text2image pipeline to free up memory
if 'text2image_pipeline' in locals() and text2image_pipeline is not None:
    del text2image_pipeline
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Text-to-image pipeline unloaded and memory cleared.")

# Load the upscaling pipeline
print("Loading upscaling pipeline...")
upscale_pipeline = StableDiffusionUpscalePipeline.from_pretrained(
    "stabilityai/stable-diffusion-x4-upscaler",
    torch_dtype=torch.float16,
    use_safetensors=True
)

# Enable CPU offload for the upscaling pipeline to manage memory
upscale_pipeline.enable_model_cpu_offload()
print("Upscaling pipeline loaded and CPU offload enabled.")

CUDA memory cache cleared for upscaler.
Text-to-image pipeline unloaded and memory cleared.
Loading upscaling pipeline...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

Upscaling pipeline loaded and CPU offload enabled.


In [50]:
import os
from PIL import Image

# Define the prompt for the upscaler (can be the same as the generation prompt or more detailed)
upscale_prompt = "beatiful landscape, village, lake on the side, full HD, 8k, details,cinematic color."

# Load the previously generated image for upscaling
# Ensure 'text_generated_image' is available, if not, load it from the saved file.
if 'text_generated_image' not in locals():
    if os.path.exists('text_generated_output.png'):
        text_generated_image = Image.open('text_generated_output.png')
        print("Loaded 'text_generated_output.png' for upscaling.")
    else:
        print("Error: 'text_generated_output.png' not found. Please run the text-to-image generation cell first.")
        # You might want to exit or handle this error more gracefully

if 'text_generated_image' in locals():
    print(f"Original image dimensions: {text_generated_image.size[0]}x{text_generated_image.size[1]}")
    print(f"Starting image upscaling with prompt: '{upscale_prompt}'")

    try:
        # Perform upscaling
        # Reduced num_inference_steps to save memory
        upscaled_image_result = upscale_pipeline(
            prompt=upscale_prompt,
            image=text_generated_image,
            num_inference_steps=30
        ).images[0]

        upscaled_output_filename = "upscaled_text_generated_output.png"
        upscaled_image_result.save(upscaled_output_filename)
        print(f"Successfully upscaled and saved '{upscaled_output_filename}'.")
        print(f"Upscaled image dimensions: {upscaled_image_result.size[0]}x{upscaled_image_result.size[1]}")

    except Exception as e:
        print(f"Error during upscaling pipeline execution: {e}")
        print("This could be due to memory issues or invalid input. Consider restarting the runtime if memory is an issue.")

Original image dimensions: 512x512
Starting image upscaling with prompt: 'beatiful landscape, village, lake on the side, full HD, 8k, details,cinematic color.'


  0%|          | 0/30 [00:00<?, ?it/s]

Error during upscaling pipeline execution: CUDA out of memory. Tried to allocate 2.00 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.36 GiB is free. Including non-PyTorch memory, this process has 13.20 GiB memory in use. Of the allocated memory 10.48 GiB is allocated by PyTorch, and 2.59 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
This could be due to memory issues or invalid input. Consider restarting the runtime if memory is an issue.


In [49]:
from google.colab import files
import os

# Download the newly upscaled image
upscaled_output_filename = "upscaled_text_generated_output.png"

if os.path.exists(upscaled_output_filename):
    files.download(upscaled_output_filename)
    print(f"'{upscaled_output_filename}' downloaded successfully.")
else:
    print(f"Error: '{upscaled_output_filename}' not found. Upscaling might have failed or not been run.")

Error: 'upscaled_text_generated_output.png' not found. Upscaling might have failed or not been run.


In [10]:
import os
print("Files in current directory:")
for file in os.listdir('.'):
    print(file)

Files in current directory:
.config
output.png
Screenshot from 2026-07-26 11-06-39.png
sample_data


In [54]:
from google.colab import files

# This will trigger a download of 'output.png' to your local machine
files.download('output.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [75]:
from google.colab import files

# This will trigger a download of 'output.png' to your local machine
# This should now work as the file has been confirmed to exist.
files.download('output.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>